# 🥭 Training YOLOv11s for Mango Counter & Ripeness Detection

This notebook trains an end-to-end **YOLOv11s** model on the **Mango Fruit Ripeness Zoner** dataset (2,327 images, 25,000+ annotations) with 3 classes:
- `Over ripe`
- `Ripe`
- `Unripe`

After training, it exports the model to **TensorFlow Lite (`.tflite`)** ready for the **MangoCounter** Flutter mobile app.

## 1. Verify GPU & Install Dependencies

In [ ]:
# Check that GPU is enabled (Go to Runtime -> Change runtime type -> T4 GPU if not active)
!nvidia-smi

# Install Ultralytics YOLO and Roboflow
!pip install -q ultralytics roboflow

## 2. Download the Dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="FegCThHwdnjnjIMZDNan")
project = rf.workspace("masa-john-phillip-d").project("mango-fruit-ripeness-zoner")
version = project.version(1)
dataset = version.download("yolov11")

print("\n✅ Dataset successfully downloaded to:", dataset.location)

## 3. Verify Dataset Configuration

In [ ]:
import yaml

yaml_path = f"{dataset.location}/data.yaml"
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Classes in dataset:", data_config.get('names'))
print("Number of classes:", data_config.get('nc'))

## 4. Train YOLOv11s

We use **YOLOv11s (Small)** with hyperparameter tuning for outdoor lighting and tree canopies.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11s checkpoint
model = YOLO("yolo11s.pt")

# Train on the dataset
results = model.train(
    data=yaml_path,
    epochs=70,
    imgsz=640,
    batch=16,
    patience=15,
    device=0,
    save=True,
    # Outdoor orchard augmentations:
    hsv_h=0.015,   # Hue jitter
    hsv_s=0.7,     # Saturation jitter for sun/shade
    hsv_v=0.4,     # Value/brightness jitter
    mosaic=1.0,    # Mosaic augmentation for small/clustered mangoes
    fliplr=0.5,    # Horizontal flip
    project="mango_runs",
    name="yolo11s_mango"
)

## 5. Evaluate Performance on Test Split

In [ ]:
from IPython.display import Image, display

# Display confusion matrix and results curve
try:
    display(Image("mango_runs/yolo11s_mango/confusion_matrix.png"))
    display(Image("mango_runs/yolo11s_mango/results.png"))
except Exception as e:
    print("Could not display graphs:", e)

# Validate on test split
metrics = model.val(split="test")
print(f"\nTest mAP@0.5: {metrics.box.map50:.4f}")
print(f"Test mAP@0.5:0.95: {metrics.box.map:.4f}")

## 6. Export to TFLite (`.tflite` for Mobile App)

In [ ]:
# Export to float32 TFLite format (compatible with tflite_flutter)
exported_path = model.export(format="tflite", imgsz=640)
print("\n✅ Exported TFLite Model:", exported_path)

## 7. Download Exported Model to PC

In [ ]:
from google.colab import files
import glob

# Locate the exported .tflite file
tflite_files = glob.glob("mango_runs/yolo11s_mango/weights/*_float32.tflite") + glob.glob("mango_runs/yolo11s_mango/weights/*.tflite")
if tflite_files:
    print(f"Downloading {tflite_files[0]} ...")
    files.download(tflite_files[0])
else:
    print("TFLite file not found in weights directory, checking root...")
    all_tflite = glob.glob("**/*.tflite", recursive=True)
    if all_tflite:
        files.download(all_tflite[0])